# Linear measurement-error regression with convMMD

This package-backed notebook fits the supported scalar errors-in-variables model



\[W=X^*+U,\quad U\sim N(0,\sigma_U^2),\]

\[Y=eta_0+eta_1X^*+arepsilon,\quad arepsilon\sim N(0,\sigma_Y^2).\]

Only `(W, Y)` and the externally known `sigma_U` are supplied to the fit. Latent values and true coefficients exist solely for the later simulation evaluation. Set `CONVMMD_NOTEBOOK_SMOKE=1` for the reduced release check.


In [ ]:
import os

import matplotlib.pyplot as plt

import torch

from convMMD import fit_measurement_error_regression

from convMMD.core.losses import mmd_laplace_kernel


## 1. Latent truth and observed data


In [ ]:
SMOKE = os.getenv("CONVMMD_NOTEBOOK_SMOKE") == "1"

SEED = 31415

DEVICE = "cuda" if torch.cuda.is_available() and not SMOKE else "cpu"

N_SAMPLES = 64 if SMOKE else 1000

STEPS = 2 if SMOKE else 1000

TRUE_INTERCEPT = -0.75

TRUE_SLOPE = 2.0

TRUE_RESIDUAL_STD = 0.5

KNOWN_MEASUREMENT_ERROR_STD = 0.8

generator = torch.Generator().manual_seed(SEED)

component = torch.rand(N_SAMPLES, generator=generator) < 0.55

latent_truth = torch.where(

    component,

    -1.35 + 0.45 * torch.randn(N_SAMPLES, generator=generator),

    1.15 + 0.65 * torch.randn(N_SAMPLES, generator=generator),

)

observed_covariate = (

    latent_truth

    + KNOWN_MEASUREMENT_ERROR_STD * torch.randn(N_SAMPLES, generator=generator)

)

response = (

    TRUE_INTERCEPT

    + TRUE_SLOPE * latent_truth

    + TRUE_RESIDUAL_STD * torch.randn(N_SAMPLES, generator=generator)

)

observed_covariate = observed_covariate.to(DEVICE)

response = response.to(DEVICE)

print(f"device={DEVICE}, smoke={SMOKE}, n={N_SAMPLES}, steps={STEPS}")


## 2. Fit from observed quantities

This cell is the complete fitting call. The public function has no parameter through which simulation truth could enter.


In [ ]:
result = fit_measurement_error_regression(

    observed_covariate,

    response,

    KNOWN_MEASUREMENT_ERROR_STD,

    steps=STEPS,

    batch_size=32 if SMOKE else 256,

    learning_rate=1e-3 if SMOKE else 3e-3,

    bandwidths=[0.5, 1.0, 2.0] if SMOKE else None,

    eval_every=max(1, STEPS // 5),

    num_blocks=1 if SMOKE else 4,

    num_bins=4,

    hidden_features=8 if SMOKE else 32,

    tail_bound=5.0,

    seed=SEED + 1,

    device=DEVICE,

    verbose=not SMOKE,

)


## 3. Parameter recovery and observed-space checks

Naive OLS is included only to show the familiar attenuation caused by covariate error. Truth is now used to evaluate parameter recovery, never to select or fit the model.


In [ ]:
centered_w = observed_covariate - observed_covariate.mean()

centered_y = response - response.mean()

naive_slope = float(

    (centered_w * centered_y).mean() / observed_covariate.var(unbiased=False)

)

naive_intercept = float(response.mean() - naive_slope * observed_covariate.mean())

_, generated_covariate, generated_response = result.sample_observed(

    N_SAMPLES, seed=SEED + 2

)

observed_pairs = torch.stack((observed_covariate, response), dim=1).cpu()

generated_pairs = torch.cat((generated_covariate, generated_response), dim=1).cpu()

forward_mmd = float(

    mmd_laplace_kernel(

        (generated_pairs - result.observed_center) / result.observed_scale,

        (observed_pairs - result.observed_center) / result.observed_scale,

        result.bandwidths,

    )

)

print(f"Truth:       intercept={TRUE_INTERCEPT:.3f}, slope={TRUE_SLOPE:.3f}, residual_std={TRUE_RESIDUAL_STD:.3f}")

print(f"Naive OLS:   intercept={naive_intercept:.3f}, slope={naive_slope:.3f}")

print(f"convMMD fit: intercept={result.intercept:.3f}, slope={result.slope:.3f}, residual_std={result.residual_std:.3f}")

print(f"Absolute slope error: {abs(result.slope - TRUE_SLOPE):.3f}")

print(f"Observed-space forward MMD: {forward_mmd:.6f}")


## 4. Essential diagnostics


In [ ]:
latent_generated = result.sample_observed(N_SAMPLES, seed=SEED + 3)[0][:, 0].cpu()

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

axes[0].scatter(

    observed_covariate.cpu(), response.cpu(), s=10, alpha=0.35, label="Observed"

)

axes[0].scatter(

    generated_covariate[:, 0].cpu(),

    generated_response[:, 0].cpu(),

    s=10,

    alpha=0.35,

    label="Forward generated",

)

axes[0].set(xlabel="Covariate", ylabel="Response", title="Observed-space check")

axes[0].legend()

axes[1].hist(latent_truth.cpu(), bins=35, density=True, alpha=0.5, label="Latent truth")

axes[1].hist(latent_generated, bins=35, density=True, alpha=0.5, label="Fitted flow")

axes[1].set(xlabel="Latent covariate", ylabel="Density", title="Simulation-only latent check")

axes[1].legend()

axes[2].plot(result.history["step"], result.history["loss"], marker="o")

axes[2].set(xlabel="Optimizer step", ylabel="MMD", title="Training diagnostic")

for ax in axes:

    ax.grid(alpha=0.2)

plt.tight_layout()

plt.show()


## Interpretation and scope

The known covariate-error standard deviation is an identification assumption. This API does not estimate it, handle replicated or multivariate covariates, or generalize the linear Gaussian response model. A small training MMD is not enough by itself; inspect forward samples and repeat substantive fits across seeds and budgets. Regression-result checkpointing is not included in this first reusable API.
